# 手撕 Iterative DPO

## 背景
Iterative DPO：多轮 DPO，每轮用当前 policy 生成新偏好对，再以当前 policy 为 reference 训练下一轮。
优势：持续自我改进，无需固定 reference model。

## 考察点
- 迭代流程：generate → label → DPO update → 重复
- 每轮 reference = 上一轮 policy
- 与 Online DPO / Self-Play 的关系

In [ ]:
import torch
import torch.nn.functional as F

def dpo_step(policy_chosen_logps, policy_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta: float = 0.1) -> torch.Tensor:
    pi_ratios = (policy_chosen_logps - ref_chosen_logps) - (policy_rejected_logps - ref_rejected_logps)
    loss = -F.logsigmoid(beta * pi_ratios).mean()
    return loss

def iterative_dpo_round(policy_logps_history, n_rounds: int = 3, beta: float = 0.1) -> torch.Tensor:
    # 模拟多轮迭代：每轮 ref = 上一轮 policy
    losses = []
    for r in range(n_rounds):
        current = policy_logps_history[r]  # (chosen_logps, rejected_logps)
        if r == 0:
            ref = (torch.zeros_like(current[0]), torch.zeros_like(current[1]))  # 初始 ref
        else:
            ref = policy_logps_history[r - 1]  # 上一轮 policy 作为 ref
        loss = dpo_step(current[0], current[1], ref[0], ref[1], beta)
        losses.append(loss.item())
    return losses

In [ ]:
# 验证迭代 DPO
torch.manual_seed(42)
n = 8
# 模拟 3 轮：每轮 chosen 概率逐渐提升
history = []
for r in range(3):
    chosen = torch.randn(n) * 2 - 1 + r * 0.5  # 逐渐提升
    rejected = torch.randn(n) * 2 - 1
    history.append((chosen, rejected))
losses = iterative_dpo_round(history, n_rounds=3, beta=0.1)
print(f"3 轮 DPO loss: {losses}")
assert all(l >= 0 for l in losses), "loss 应非负"
print("✅ Iterative DPO 多轮训练验证通过")